# 图像编码 · 学生版 notebook

用于课堂跟随观察。隐藏部分教师验证逻辑，只保留必要预测与运行结果。

## 0. 运行位置检查

In [ ]:

from pathlib import Path
lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"
assert assets_dir.is_dir(), f"请从课程目录启动 notebook。当前目录：{lesson_dir}"
print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)


## Q1 · Sampling

先预测：如果显示尺寸一样，降低 pixel dimensions 后，哪些细节最先消失？

In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw

path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)

w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)

draw = ImageDraw.Draw(img)
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)

img.save(path)
assert img.size == (256, 256)
assert img.mode == "RGB"
print("created:", path)


from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
src = Image.open(source_path).convert("RGB")
sizes = [256, 64, 32, 16, 8]
sampling_outputs = {}

for n in sizes:
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)
    assert enlarged.mode == "RGB"
    assert enlarged.size == (512, 512)
    sampling_outputs[n] = enlarged
    print(f"{n} × {n}")
    display(enlarged)

def show_sampling(n: int):
    """课堂即时验证：输入学生提出的新尺寸。"""
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)
    print(f"{n} × {n}")
    display(enlarged)
    return enlarged


## Q3 · Quantization

先预测：pixel dimensions 不变，只把整幅图最多颜色数限制为 16 或 2，红绿接缝还会一直看得见吗？

In [ ]:

from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}

for n in targets:
    q = src.quantize(
        colors=n,
        method=Image.Quantize.MEDIANCUT,
        dither=Image.Dither.NONE,
    ).convert("RGB")
    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)
    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n
    outputs[n] = q

print("original")
display(src)
for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])


## Q3 · 接缝证据

先写下你的解释，再运行：16 colors 和 2 colors 的接缝两侧是否相同？

In [ ]:

seam_left = (72, 40)
seam_right = (73, 40)

for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)
    print(
        f"{n:2d} colors | "
        f"left={left_rgb} | right={right_rgb} | "
        f"same={left_rgb == right_rgb}"
    )


## Q6 · RGB → bits

运行前先写下预测：

- 255 的 8-bit binary 是什么？
- 128 的 8-bit binary 是什么？
- 0 的 8-bit binary 是什么？

运行后比较你的预测。

In [ ]:

def to_8bit(n: int) -> str:
    return format(n, "08b")

rgb = (255, 128, 0)
print("RGB =", rgb)
print("R", rgb[0], "→", to_8bit(rgb[0]))
print("G", rgb[1], "→", to_8bit(rgb[1]))
print("B", rgb[2], "→", to_8bit(rgb[2]))
print("combined:", " ".join(to_8bit(v) for v in rgb))


## Q7 · BMP 中像素数据之前有什么？

In [ ]:

from pathlib import Path
import struct

path = Path("assets/q7-3x2-24bit.bmp")
data = path.read_bytes()
print("signature:", data[:2])
print("file bytes:", len(data))
print("pixel-data offset:", struct.unpack_from("<I", data, 10)[0])
print("bits per pixel:", struct.unpack_from("<H", data, 28)[0])
print("top red pixel bytes at 0x42:", data[66:69].hex(" "))
